# Databricks / PySpark for Data Analysts/Engineers — Module 15: Final Project

This is the course's last module -- a complete, runnable **end-to-end data pipeline**
bringing together everything we learned in Modules 1-14: loading raw, "dirty" data,
cleaning and transforming it with the DataFrame API, enriching it with joins and window
functions, analyzing it with Spark SQL, training and evaluating an MLlib model, and
writing the results out as partitioned Parquet. We'll organize the whole thing in the
spirit of the **Medallion Architecture** from Module 11: bronze (raw data) -> silver
(cleaned and joined) -> gold (ready for analysis and ML).

## Table of Contents
1. [Bronze: simulating "dirty" raw data](#sec1)
2. [Silver: cleaning and transformations (DataFrame API)](#sec2)
3. [Silver: enrichment -- joins and window functions](#sec3)
4. [Gold: business analysis (Spark SQL)](#sec4)
5. [Gold: an MLlib model -- predicting order cancellation](#sec5)
6. [Writing the results -- partitioned Parquet](#sec6)
7. [Course summary and final exercises](#sec7)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

# Extra JVM options (--add-opens) are needed when running Spark locally on Java 17+ and
# using pandas UDFs / Apache Arrow (Module 8) -- newer JDKs block, by default, the
# reflective off-heap memory access that Arrow needs. In Databricks you don't need to
# set this by hand -- the platform handles it for you.
_jvm_options = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("spark_course")
    .config("spark.driver.extraJavaOptions", _jvm_options)
    .config("spark.executor.extraJavaOptions", _jvm_options)
    .getOrCreate()
)
spark


In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")


> ⚡ **Why we lower shuffle partitions**
>
> As we learned in Modules 9 and 13, the default 200 shuffle partitions is heavy overhead on a small, local dataset. We set it to 8 at the top of the WHOLE notebook -- in a real production environment (Databricks, a large cluster) we'd leave the default or tune it to the cluster's size.

<a id="sec1"></a>
## 1. Bronze: simulating "dirty" raw data

In a real project, input data is rarely perfect -- there are duplicates, missing values,
inconsistent casing. Let's simulate that: we'll generate our familiar online store
dataset, deliberately "dirty" it, write it out as raw CSV files (the **bronze** layer),
and then read it back in -- exactly how a real data ingestion process would look.

In [ ]:
import numpy as np
import pandas as pd
import os

rng = np.random.default_rng(42)

# --- generate "clean" data, just like throughout the course -----------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=[0.35, 0.45, 0.20]),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}
product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2) for c in product_categories],
})

n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values
cancel_prob = np.where(order_segment == "New", 0.22, np.where(order_segment == "Standard", 0.10, 0.04))
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(cancelled == 1, "Cancelled", rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]))
base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]
for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id, "order_id": int(order_id),
            "product_id": int(product_id), "quantity": quantity,
            "unit_price": float(product_price[product_id]), "discount": float(discount),
        })
        item_id += 1
order_items_pd = pd.DataFrame(item_rows)

# --- now we deliberately "dirty" the data, as if it came from a real source -
customers_dirty = customers_pd.copy()
# inconsistent casing in cities
case_mask = rng.random(len(customers_dirty)) < 0.15
customers_dirty.loc[case_mask, "city"] = customers_dirty.loc[case_mask, "city"].str.upper()
# duplicate rows (a common bug from repeated ingestion)
duplicates = customers_dirty.sample(10, random_state=1)
customers_dirty = pd.concat([customers_dirty, duplicates], ignore_index=True)
# missing values in segment
missing_mask = rng.random(len(customers_dirty)) < 0.05
customers_dirty.loc[missing_mask, "segment"] = None

items_dirty = order_items_pd.copy()
# a few invalid (negative) quantities -- a source-system bug
negative_mask = rng.random(len(items_dirty)) < 0.01
items_dirty.loc[negative_mask, "quantity"] = -items_dirty.loc[negative_mask, "quantity"]

os.makedirs("bronze", exist_ok=True)
customers_dirty.to_csv("bronze/customers_raw.csv", index=False)
products_pd.to_csv("bronze/products_raw.csv", index=False)
orders_pd.to_csv("bronze/orders_raw.csv", index=False)
items_dirty.to_csv("bronze/order_items_raw.csv", index=False)

print(f"Wrote raw CSV files. Customers (with duplicates): {len(customers_dirty)} rows (originally {n_customers}).")


In [ ]:
# We read the raw data as a real ingestion process would -- with an explicit schema
# (Module 6), to avoid Spark guessing the types
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("days_since_signup", IntegerType(), True),
])

customers_bronze = spark.read.csv("bronze/customers_raw.csv", header=True, schema=customers_schema)
products_bronze = spark.read.csv("bronze/products_raw.csv", header=True, inferSchema=True)
orders_bronze = spark.read.csv("bronze/orders_raw.csv", header=True, inferSchema=True)
items_bronze = spark.read.csv("bronze/order_items_raw.csv", header=True, inferSchema=True)

print(f"customers_bronze: {customers_bronze.count()} rows")
customers_bronze.show(5)


<a id="sec2"></a>
## 2. Silver: cleaning and transformations (DataFrame API)

Now we apply exactly the DataFrame API operations we learned in Modules 2-3: removing
duplicates, handling missing values, standardizing text, and filtering out invalid
rows.

In [ ]:
from pyspark.sql import functions as F

customers_silver = (
    customers_bronze
    .dropDuplicates(["customer_id"])                                # remove duplicate rows
    .withColumn("city", F.initcap(F.lower(F.col("city"))))           # standardize casing
    .fillna({"segment": "Standard"})                                 # reasonable default for missing values
)

print(f"After cleaning: {customers_silver.count()} rows (was {customers_bronze.count()} before de-duplication).")
customers_silver.groupBy("city").count().orderBy("city").show()


In [ ]:
# Filter out invalid order items (negative quantity = a source data bug)
count_before = items_bronze.count()
items_silver = items_bronze.filter(F.col("quantity") > 0)
count_after = items_silver.count()

print(f"Dropped {count_before - count_after} invalid order items (negative quantity).")


> ⚠️ **Always measure what you drop**
>
> In a real data pipeline, every cleaning step should log HOW MANY rows it dropped or changed -- otherwise a silent drop in data quality (e.g. a source system starts sending 30% invalid records) would go unnoticed. The `print()` calls above are a simplified version of this kind of data-quality monitoring.

<a id="sec3"></a>
## 3. Silver: enrichment — joins and window functions

We join the cleaned tables (Module 4) and add columns computed with window functions
(Module 7) -- in this case each order's total value and the customer's order number in
chronological order (a useful signal: "is this the customer's first order?").

In [ ]:
from pyspark.sql import Window

order_values = (
    items_silver.groupBy("order_id")
    .agg(
        F.sum(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))).alias("value"),
        F.count("*").alias("item_count"),
        F.avg("unit_price").alias("avg_price"),
    )
)

orders_silver = (
    orders_bronze.join(order_values, "order_id", "inner")
    .join(customers_silver, "customer_id", "inner")
    .withColumn("is_cancelled", (F.col("status") == "Cancelled").cast("int"))
)

customer_window = Window.partitionBy("customer_id").orderBy("order_date")
orders_silver = orders_silver.withColumn(
    "customer_order_number", F.row_number().over(customer_window)
)

orders_silver.select(
    "order_id", "customer_id", "segment", "value", "customer_order_number", "is_cancelled"
).show(10)


In [ ]:
orders_silver.cache()
print(f"orders_silver: {orders_silver.count()} rows (after cache -- later references are faster, Module 9).")


<a id="sec4"></a>
## 4. Gold: business analysis (Spark SQL)

We register the enriched table as a temp view (Module 5) and answer concrete business
questions with plain SQL -- exactly as an analyst would, without needing to know the
DataFrame API syntax at all.

In [ ]:
orders_silver.createOrReplaceTempView("orders_view")

segment_report = spark.sql("""
    SELECT
        segment,
        COUNT(*) AS order_count,
        ROUND(AVG(value), 2) AS avg_value,
        ROUND(100.0 * SUM(is_cancelled) / COUNT(*), 1) AS pct_cancelled
    FROM orders_view
    GROUP BY segment
    ORDER BY avg_value DESC
""")
segment_report.show()


In [ ]:
city_report = spark.sql("""
    SELECT
        city,
        COUNT(*) AS order_count,
        ROUND(SUM(value), 2) AS total_revenue
    FROM orders_view
    WHERE status != 'Cancelled'
    GROUP BY city
    ORDER BY total_revenue DESC
""")
city_report.show()


> ⚡ **DataFrame API or Spark SQL -- this module uses both**
>
> Section 3 used the DataFrame API (joins, window functions), and section 4 used plain SQL on that same, enriched DataFrame. This is best practice on real projects: transformations/feature engineering in the DataFrame API, business reporting and exploration in SQL -- both APIs share the same execution engine (Module 5).

<a id="sec5"></a>
## 5. Gold: an MLlib model — predicting order cancellation

We train a binary classification model (Modules 12-13) that predicts whether an order
will be cancelled, based on features available AT THE MOMENT the order is placed
(customer segment, city, item count, average price, the customer's order sequence
number) -- we deliberately do NOT use `status` or `value` itself as an input feature, to
avoid leaking information from the future (*data leakage*).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

train, test = orders_silver.randomSplit([0.8, 0.2], seed=42)

indexer_segment = StringIndexer(inputCol="segment", outputCol="segment_idx", handleInvalid="keep")
indexer_city = StringIndexer(inputCol="city", outputCol="city_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["segment_idx", "city_idx"], outputCols=["segment_ohe", "city_ohe"])

assembler = VectorAssembler(
    inputCols=["segment_ohe", "city_ohe", "item_count", "avg_price", "customer_order_number"],
    outputCol="features",
)
model_lr = LogisticRegression(featuresCol="features", labelCol="is_cancelled")

pipeline = Pipeline(stages=[indexer_segment, indexer_city, encoder, assembler, model_lr])
model = pipeline.fit(train)

predictions = model.transform(test)
evaluator = BinaryClassificationEvaluator(labelCol="is_cancelled", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"Test set AUC-ROC: {auc:.3f}")


> 🧱 **This is the same pattern as Modules 12-13**
>
> A Pipeline (StringIndexer -> OneHotEncoder -> VectorAssembler -> model), a train/test split, and BinaryClassificationEvaluator are exactly the same building blocks as in Modules 12-13 -- here we're just assembling them in the context of a full, real pipeline, instead of an isolated example.

<a id="sec6"></a>
## 6. Writing the results — partitioned Parquet

Finally, we write TWO result tables to the **gold** layer as Parquet, partitioned by
`segment` (Module 6) -- a columnar format ready for fast analysis in BI tools or
downstream notebooks: (1) the model's predictions, (2) the segment report from
section 4.

In [ ]:
predictions.select(
    "order_id", "customer_id", "segment", "city", "value", "is_cancelled", "prediction"
).write.mode("overwrite").partitionBy("segment").parquet("gold/cancellation_predictions")

segment_report.write.mode("overwrite").parquet("gold/segment_report")

print("Wrote results to gold/cancellation_predictions (partitioned by segment) and gold/segment_report.")


In [ ]:
# Verification: reading with partition pruning -- Spark reads ONLY the "Premium" partition
check = spark.read.parquet("gold/cancellation_predictions").filter(F.col("segment") == "Premium")
check.explain()
print(f"Number of rows in the Premium partition: {check.count()}")


<a id="sec7"></a>
## 7. Course summary and final exercises

We built a complete data pipeline in the spirit of the Medallion Architecture:

- **bronze** -- raw, "dirty" CSV data, read with an explicit schema,
- **silver** -- cleaned (duplicates, missing values, standardization), enriched with
  joins and window functions, cached in memory,
- **gold** -- business reports (Spark SQL), an MLlib model (Pipeline + AUC-ROC
  evaluation), and final results written out as partitioned Parquet.

This closes out the entire Databricks/PySpark course -- from `SparkSession` in Module 1,
through the DataFrame API, joins, Spark SQL, reading/writing data, window functions,
UDFs, optimization, Delta Lake, the Lakehouse architecture, MLlib, all the way to working
in a real Databricks Workspace in Module 14.

### 📝 Exercise 1: Add another feature

Add a new feature to the model from section 5: the customer's `days_since_signup` (from the `customers_silver` table, which you'll need to join into the feature set before training). Check whether the AUC-ROC improved.

<details>
<summary>Show solution</summary>

```python
assembler2 = VectorAssembler(
    inputCols=["segment_ohe", "city_ohe", "item_count", "avg_price",
               "customer_order_number", "days_since_signup"],
    outputCol="features",
)
pipeline2 = Pipeline(stages=[indexer_segment, indexer_city, encoder, assembler2, model_lr])
model2 = pipeline2.fit(train)
predictions2 = model2.transform(test)
auc2 = evaluator.evaluate(predictions2)
print(f"AUC-ROC with the extra feature: {auc2:.3f} (was {auc:.3f})")
```

</details>

### 📝 Exercise 2: A monthly report -- extending the pipeline

This module's dataset doesn't include an `employees` table, but it was present in Modules 4-5. Write a Spark SQL query on `orders_view` that computes total revenue and order count PER MONTH (use `date_format(order_date, 'yyyy-MM')`).

<details>
<summary>Show solution</summary>

```python
monthly_report = spark.sql("""
    SELECT
        date_format(order_date, 'yyyy-MM') AS month,
        COUNT(*) AS order_count,
        ROUND(SUM(value), 2) AS total_revenue
    FROM orders_view
    WHERE status != 'Cancelled'
    GROUP BY date_format(order_date, 'yyyy-MM')
    ORDER BY month
""")
monthly_report.show(24)
```

</details>

### 📝 Exercise 3: Read the saved results back

Read `gold/segment_report` (written in section 6) back in as a new DataFrame and display it -- confirm that the data after writing and reading matches the original.

<details>
<summary>Show solution</summary>

```python
segment_report_read = spark.read.parquet("gold/segment_report")
segment_report_read.orderBy(F.col("avg_value").desc()).show()
```

</details>

> 🔥 **Final challenge: a full report with a data-quality alert**
>
> Extend the cleaning step (section 2) so that BEFORE dropping the invalid order items, it computes the percentage of dropped rows. If it exceeds 5%, print a warning (`print("WARNING: ...")`) -- a simulation of a simple data-quality alert that a real pipeline would turn into a notification (Module 14, `dbutils` + Jobs). Finally, print a one-sentence summary of the WHOLE pipeline: how many orders made it to the gold layer, and what the model's final AUC-ROC was.

<details>
<summary>Show solution</summary>

```python
pct_dropped = 100.0 * (count_before - count_after) / count_before
if pct_dropped > 5.0:
    print(f"WARNING: dropped {pct_dropped:.1f}% of order items -- check the data source!")
else:
    print(f"Data quality OK: only {pct_dropped:.2f}% of order items were dropped.")

print(
    f"Pipeline summary: {orders_silver.count()} orders reached the gold layer, "
    f"final model AUC-ROC = {auc:.3f}."
)
```

In a real Databricks setup this alert would trigger a notification through Jobs (Module 14) instead of print() -- for example halting the pipeline and sending an email/Slack message when data quality drops below the threshold.

</details>

## Congratulations!

This is the end of the Databricks / PySpark course -- and of the whole series that began
with Python, continued through SQL/PostgreSQL and Statistics. You've gone from single
SQL queries and `pandas.DataFrame`, through statistical tests and regression, all the
way to distributed processing of petabytes of data and cluster-scale machine learning.
Good luck on your real-world data projects!